# desafio ciia

**Status:** Em andamento · **Área:** classificacao-tweets

Pipeline de limpeza, TF-IDF e regressão logística para tweets; acurácia/relatório calculados no próprio treino.

Este arquivo está identificado como **Em andamento**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


# Regras

Entendendo o desafio:

Precisarei de 3 arquivos:

1.   train.csv
2.   test.csv
3.   sample_submission.csv





Precisarei tratar ambos


1.   Nulos
2.   Duplicados
3.   Maiusculos e Minusculos



Conjuntos e Tipos:


1.   train.csv - o conjunto de treinamento
2.   test.csv - o conjunto de testes
1.   sample_submission.csv - um arquivo de envio de amostra no formato correto




Colunas:
*   id- um identificador único para cada tweet
*   text - o texto do tweet
*   location - o local de onde o tweet foi enviado (pode estar em branco)
*   keyword - uma palavra-chave específica do tweet (pode estar em branco)
*target- somente em train.csv , isso denota se um tweet é sobre um desastre real ( 1) ou não ( 0)


primeiro, transformar os dados dos tweets em dado numéricos que uma IA possa utilizar como entrada.

bibliotecas do python:
*   NLTK
*   Spacy



estruturar um modelo de inteligência artificial para ser treinado utilizando os textos transformados em entradas, e as classificações em saída.


Última dica: Tente usar bibliotecas estabelecidas, como o Keras, TensorFlow, PyTorch ou mesmo a biblioteca de Transformers do Hugging Face!


# Leitura

In [ ]:
import zipfile
import os
import pandas as pd

caminho_zip = "/content/nlp-getting-started.zip"
pasta_destino = "/content/arquivo"


with zipfile.ZipFile(caminho_zip, "r") as zip_referencia:
  zip_referencia.extractall(pasta_destino)

  print(f"Arquivos dentro do zip: {zip_referencia.namelist()}")
  print(f"Arquivo extraidos: {pasta_destino}")


In [ ]:
df_submission = pd.read_csv("/content/arquivo/sample_submission.csv")
print(df_submission.info())
print(df_submission.head())

In [ ]:
df_test = pd.read_csv("/content/arquivo/test.csv")
print(df_test.info())
print(df_test.head())

In [ ]:
df_train = pd.read_csv("/content/arquivo/train.csv")
print(df_train.info())
print(df_train.head())

# Visualização nulos

In [ ]:
df_submission.isnull().sum()

In [ ]:
df_test.isnull().sum()

In [ ]:
df_train.isnull().sum()

# Tratamento Nulos



In [ ]:
df_test["location"].fillna("desconhecido", inplace=True)
df_test["keyword"].fillna("desconhecido", inplace=True)

In [ ]:
df_train["keyword"].fillna("desconhecido", inplace=True)
df_train["location"].fillna("desconhecido", inplace=True)

# Tratando Duplicados

In [ ]:
df_submission.drop_duplicates(inplace=True)
df_test.drop_duplicates(inplace=True)
df_train.drop_duplicates(inplace=True)

# Normalização de caracteres

In [ ]:
df_submission.info()

In [ ]:
df_test.info()
#1   keyword   3263 non-null   object
#2   location  3263 non-null   object
#3   text      3263 non-null   object

In [ ]:
df_train.info()
# 1   keyword   7613 non-null   object
# 2   location  7613 non-null   object
# 3   text      7613 non-null   object

Tranformando objetos em string para normalizar - acentos, espaços, numeros, maiusculas e minusculas

In [ ]:
!pip install unidecode

In [ ]:
from unidecode import unidecode

dataframes_tratamento = [df_train, df_test]

for df in dataframes_tratamento:
  for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.lower()
    df[col] = df[col].apply(unidecode)
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(r"[^\w\s]", "", regex=True)


In [ ]:
df_test.info()
df_train.info()

# Tokenização

Download dos Stopwords da lingua portuguesa-Remoção de Stopwords

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stopwords_en = set(stopwords.words('english'))
df_train['tokens'] = df_train['text'].apply(lambda x: str(x).lower().split())
df_test['tokens'] = df_test['text'].apply(lambda x: str(x).lower().split())

df_train['tokens_sem_stopwords'] = df_train['tokens'].apply(
    lambda tokens: [palavra for palavra in tokens if palavra not in stopwords_en]
)
df_test['tokens_sem_stopwords'] = df_test['tokens'].apply(
    lambda tokens: [palavra for palavra in tokens if palavra not in stopwords_en]
)

Exemplo de como a tokenização funcionou

In [ ]:
df_train[['text', 'tokens', 'tokens_sem_stopwords']].sample(5)


# Vetorização

Usaremos o IDF - Raridade de palavras em outros Tweets - Frequencias Unicas

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
# Junta os tokens em strings novamente
df_train['texto_final'] = df_train['tokens_sem_stopwords'].apply(lambda x: ' '.join(x))
df_test['texto_final'] = df_test['tokens_sem_stopwords'].apply(lambda x: ' '.join(x))


In [ ]:
# Cria o vetor de TF-IDF
vetorizador = TfidfVectorizer()

# Ajusta e transforma os textos de treino
X_train_tfidf = vetorizador.fit_transform(df_train['texto_final'])

# Apenas transforma o teste (sem ajustar de novo!)
X_test_tfidf = vetorizador.transform(df_test['texto_final'])


# Modelo de Regressão

In [ ]:
# Importa a biblioteca do modelo
from sklearn.linear_model import LogisticRegression

# Define os dados de entrada e saída
X = X_train_tfidf
y = df_train['target']

# Cria o modelo
modelo = LogisticRegression(max_iter=1000)

# Treina (ajusta) o modelo com os dados
modelo.fit(X, y)


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Faz previsões com os dados de treino
y_pred = modelo.predict(X)

# Avalia o desempenho
print("Acurácia:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))


In [ ]:
# Aplica o modelo treinado no dataset de teste
y_test_pred = modelo.predict(X_test_tfidf)

# Adiciona as previsões no df_test
df_test['target_pred'] = y_test_pred

# Visualiza alguns resultados
df_test[['text', 'target_pred']].sample(5)


## Continuidade

Criar validação estratificada antes de ajustar TF-IDF, reportar F1 e matriz de confusão fora do treino; preservar IDs e cardinalidade de test/submission, gerar arquivo de envio e documentar fonte/regras. Corrigir texto: stopwords efetivamente em inglês.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.